<a href="https://colab.research.google.com/github/mf2056/Dissertation/blob/main/Baseline_UNet.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import numpy as np
import os
import tensorflow as tf
from tensorflow.keras import layers, models

In [3]:
# Load the data
X_train = np.load('/content/drive/MyDrive/X_train.npy')
Y_train_mask = np.load('/content/drive/MyDrive/Y_train_mask.npy')
X_test = np.load('/content/drive/MyDrive/X_test.npy')
Y_test_mask = np.load('/content/drive/MyDrive/Y_test_mask.npy')

In [4]:
import numpy as np
import tensorflow as tf

# Define the ESA class map
class_map = {
    10: 0, #(Tree cover, "#006400")
    20: 1, #(Shrubland, "#ffbb22")
    30: 2, #(Grassland, "#ffff4c")
    40: 3, #(Cropland, "#f096ff")
    60: 3, #(Bare / Sparse vegetation, "#b4b4b4")  Combined with Cropland
    50: 4, #(Built-up, "#fa0000")
    80: 5, #(Permanent water bodies, "#0064ff")
    90: 5, #(Herbaceous wetland, "#0096a0")   Combined with Water Bodies
}

# map the high ESA numbers to 0, 1, 2, 3, 4, 5 across all pixels
lut = np.full(91, -1, dtype=np.int32)
for old_id, new_id in class_map.items():
    lut[old_id] = new_id

# Apply mapping to the whole 3D array (Spatial Mapping)
Y_train_ready = lut[Y_train_mask]
Y_test_ready = lut[Y_test_mask]

print(f"Unique IDs in merged train mask: {np.unique(Y_train_ready)}")

Unique IDs in merged train mask: [0 1 2 3 4 5]


In [5]:
num_classes = 6
Y_train_cat = tf.keras.utils.to_categorical(Y_train_ready, num_classes=num_classes)
Y_test_cat = tf.keras.utils.to_categorical(Y_test_ready, num_classes=num_classes)

print(f"X_train shape: {X_train.shape}") # no.of bands(7)
print(f"Y_train shape: {Y_train_cat.shape}") # no.of classes(6)

X_train shape: (639, 256, 256, 7)
Y_train shape: (639, 256, 256, 6)


In [6]:
# Converting for RAM efficiency

X_train = X_train.astype('float32')
Y_train_cat = Y_train_cat.astype('float32')
X_test = X_test.astype('float32')
Y_test_cat = Y_test_cat.astype('float32')

In [7]:
import tensorflow as tf
from tensorflow.keras import layers, models

def basic_unet(input_shape=(256, 256, 3), num_classes=6):
    inputs = layers.Input(input_shape)

    # ENCODER
    c1 = layers.Conv2D(64, (3, 3), activation='relu', padding='same')(inputs)
    c1 = layers.Conv2D(64, (3, 3), activation='relu', padding='same')(c1)
    p1 = layers.MaxPooling2D((2, 2))(c1) # 128x128

    c2 = layers.Conv2D(128, (3, 3), activation='relu', padding='same')(p1)
    c2 = layers.Conv2D(128, (3, 3), activation='relu', padding='same')(c2)
    p2 = layers.MaxPooling2D((2, 2))(c2) # 64x64

    # BRIDGE
    c3 = layers.Conv2D(256, (3, 3), activation='relu', padding='same')(p2)
    c3 = layers.Conv2D(256, (3, 3), activation='relu', padding='same')(c3)

    # DECODER
    u4 = layers.Conv2DTranspose(128, (2, 2), strides=(2, 2), padding='same')(c3)
    u4 = layers.concatenate([u4, c2])
    c4 = layers.Conv2D(128, (3, 3), activation='relu', padding='same')(u4)
    c4 = layers.Conv2D(128, (3, 3), activation='relu', padding='same')(c4)

    u5 = layers.Conv2DTranspose(64, (2, 2), strides=(2, 2), padding='same')(c4)
    u5 = layers.concatenate([u5, c1])
    c5 = layers.Conv2D(64, (3, 3), activation='relu', padding='same')(u5)
    c5 = layers.Conv2D(64, (3, 3), activation='relu', padding='same')(c5)

    # --- OUTPUT LAYER ---
    # Softmax for multi-class segmentation
    outputs = layers.Conv2D(num_classes, (1, 1), activation='softmax')(c5)

    model = models.Model(inputs=[inputs], outputs=[outputs])
    return model

# Initialize
model = basic_unet()

In [9]:
import tensorflow as tf
import gc

# Compile model
model = basic_unet(input_shape=(256, 256, 7), num_classes=6)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# Clear the Keras session
tf.keras.backend.clear_session()
gc.collect()

# Start training
history = model.fit(
    X_train, Y_train_cat,
    validation_data=(X_test, Y_test_cat),
    epochs=20,
    batch_size=16,
    verbose=1
)

Epoch 1/20
40/40 ━━━━━━━━━━━━━━━━━━━━ 145s 2s/step - accuracy: 0.4257 - loss: 1.3995 - val_accuracy: 0.5796 - val_loss: 1.0109
Epoch 2/20
40/40 ━━━━━━━━━━━━━━━━━━━━ 24s 583ms/step - accuracy: 0.5648 - loss: 1.0171 - val_accuracy: 0.6361 - val_loss: 0.8500
Epoch 3/20
40/40 ━━━━━━━━━━━━━━━━━━━━ 22s 551ms/step - accuracy: 0.6285 - loss: 0.8864 - val_accuracy: 0.6239 - val_loss: 0.8997
Epoch 4/20
40/40 ━━━━━━━━━━━━━━━━━━━━ 22s 555ms/step - accuracy: 0.6441 - loss: 0.8363 - val_accuracy: 0.6866 - val_loss: 0.7365
Epoch 5/20
40/40 ━━━━━━━━━━━━━━━━━━━━ 23s 582ms/step - accuracy: 0.6688 - loss: 0.7856 - val_accuracy: 0.7050 - val_loss: 0.7104
Epoch 6/20
40/40 ━━━━━━━━━━━━━━━━━━━━ 22s 552ms/step - accuracy: 0.6785 - loss: 0.7755 - val_accuracy: 0.7003 - val_loss: 0.7182
Epoch 7/20
40/40 ━━━━━━━━━━━━━━━━━━━━ 23s 583ms/step - accuracy: 0.6848 - loss: 0.7658 - val_accuracy: 0.7020 - val_loss: 0.7143
Epoch 8/20
40/40 ━━━━━━━━━━━━━━━━━━━━ 22s 555ms/step - accuracy: 0.6937 - loss: 0.7437 - val_accura

In [10]:
import numpy as np
from sklearn.metrics import classification_report, accuracy_score, f1_score
import gc

def pixels_to_patch_labels(y_array):
    """ Converts (N, 256, 256) pixel masks into (N,) majority-class labels. """
    return np.array([np.argmax(np.bincount(patch.flatten(), minlength=6)) for patch in y_array])

print("Predicting U-Net Masks...")
# 1. Generate Pixel Predictions (Output is N, 256, 256)
test_preds_raw = model.predict(X_test, batch_size=8, verbose=1)
test_preds_pixels = np.argmax(test_preds_raw, axis=-1)

train_preds_raw = model.predict(X_train, batch_size=8, verbose=1)
train_preds_pixels = np.argmax(train_preds_raw, axis=-1)

# Free up RAM immediately
del test_preds_raw, train_preds_raw
gc.collect()

# 2. Convert to Patch-Level (Majority Class)
y_train_patch_true = pixels_to_patch_labels(Y_train_ready)
y_train_patch_pred = pixels_to_patch_labels(train_preds_pixels)

y_test_patch_true = pixels_to_patch_labels(Y_test_ready)
y_test_patch_pred = pixels_to_patch_labels(test_preds_pixels)

# 3. Calculate Metrics
train_acc = accuracy_score(y_train_patch_true, y_train_patch_pred)
test_acc = accuracy_score(y_test_patch_true, y_test_patch_pred)
test_f1_macro = f1_score(y_test_patch_true, y_test_patch_pred, average="macro")

# 4. Results Display
merged_names = ["Tree cover", "Shrubland", "Grassland", "Cropland", "Built-up", "Water"]

print(f"Train Accuracy (Patch): {train_acc:.4f}")
print(f"Test Accuracy (Patch):  {test_acc:.4f}")
print(f"Test Macro F1 (Patch):  {test_f1_macro:.4f}")

print("\n CLASSIFICATION REPORT ")
print(classification_report(y_test_patch_true, y_test_patch_pred, target_names=merged_names))

Predicting U-Net Masks...
16/16 ━━━━━━━━━━━━━━━━━━━━ 21s 726ms/step
80/80 ━━━━━━━━━━━━━━━━━━━━ 18s 231ms/step
Train Accuracy (Patch): 0.8153
Test Accuracy (Patch):  0.8240
Test Macro F1 (Patch):  0.8247

 CLASSIFICATION REPORT 
              precision    recall  f1-score   support

  Tree cover       0.75      1.00      0.86         6
   Shrubland       0.65      0.85      0.73        13
   Grassland       0.62      0.64      0.63        28
    Cropland       1.00      0.76      0.86        29
    Built-up       0.86      0.86      0.86        22
       Water       1.00      1.00      1.00        27

    accuracy                           0.82       125
   macro avg       0.81      0.85      0.82       125
weighted avg       0.84      0.82      0.83       125

